# Google Colab Setup for Sign Language Translation

This notebook helps you run your `training_dataset` project on Google Colab with GPU, use Google Drive for persistence, and save outputs/checkpoints.

Run cells in order.

## 1) Verify Colab Runtime and Python Version
Run this first to confirm you're actually in Colab and see Python/runtime details.

In [ ]:
import sys, os, platform

print('Python:', sys.version)
print('Platform:', platform.platform())

try:
    import google.colab
    print('Running in Google Colab: YES')
except ImportError:
    print('Running in Google Colab: NO (open this notebook in colab.research.google.com)')

print('Current working directory:', os.getcwd())

## 2) Install and Import Required Packages
Install dependencies used by your project (`train_slt.py`, `slt_dataset.py`, `slt_model.py`).

In [ ]:
# In Colab, this installs quickly on fresh runtimes
!pip -q install numpy pandas tqdm opencv-python

# Install PyTorch (CUDA build for Colab)
!pip -q install torch torchvision --index-url https://download.pytorch.org/whl/cu121

import numpy as np
import pandas as pd
import torch
import cv2
from tqdm import tqdm

print('numpy:', np.__version__)
print('pandas:', pd.__version__)
print('torch:', torch.__version__)
print('opencv:', cv2.__version__)

## 3) Mount Google Drive
Authenticate and mount Google Drive to read/write persistent files.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive'
PROJECT_DIR = '/content/training_dataset'
print('Drive mounted. DRIVE_ROOT =', DRIVE_ROOT)
print('PROJECT_DIR =', PROJECT_DIR)

## 4) Load Data from Drive
Copy your project folder from Drive to Colab local storage for faster training, then verify required files.

In [ ]:
import os
import shutil

# Update this if your folder name/path differs in Drive
DRIVE_PROJECT = '/content/drive/MyDrive/training_dataset'
LOCAL_PROJECT = '/content/training_dataset'

if os.path.exists(LOCAL_PROJECT):
    shutil.rmtree(LOCAL_PROJECT)

shutil.copytree(DRIVE_PROJECT, LOCAL_PROJECT)
print('Copied project to', LOCAL_PROJECT)

required = ['train_slt.py', 'slt_model.py', 'slt_dataset.py', 'feature_extractor.py']
for f in required:
    p = os.path.join(LOCAL_PROJECT, f)
    print(f, '->', 'FOUND' if os.path.exists(p) else 'MISSING')

os.chdir(LOCAL_PROJECT)
print('Now in:', os.getcwd())

## 5) Run a Sample Data Processing Task
Validate pipeline by loading TSV metadata and a sample feature file.

In [ ]:
import pandas as pd
import numpy as np
from glob import glob

train_tsv = 'how2sign/tsv_files_how2sign/tsv_files_how2sign/cvpr23.fairseq.i3d.train.how2sign.tsv'
train_feat_dir = 'how2sign/i3d_features_how2sign/i3d_features_how2sign/train'

df = pd.read_csv(train_tsv, sep='\t')
print('TSV rows:', len(df))
print('Columns:', list(df.columns)[:8], '...')

feat_files = glob(f'{train_feat_dir}/*.npy')
print('Feature files found:', len(feat_files))

if feat_files:
    sample = np.load(feat_files[0])
    print('Sample feature shape:', sample.shape, 'dtype:', sample.dtype)
    # Simple processing check
    mean_per_dim = sample.mean(axis=0)
    print('Processed output shape (mean over time):', mean_per_dim.shape)

## 6) Enable and Verify GPU
In Colab: Runtime → Change runtime type → Hardware accelerator = GPU, then run this cell.

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    x = torch.randn(4096, 4096, device='cuda')
    y = torch.randn(4096, 4096, device='cuda')
    z = x @ y
    print('GPU test tensor shape:', z.shape)
else:
    print('No GPU found. Switch runtime to GPU for fast training.')

## Training Commands (Quick Test + Full Run)
Use quick mode first to verify everything works, then run full training.

In [ ]:
# Quick sanity run (small model + few epochs)
!python train_slt.py --device cuda --batch_size 8 --d_model 256 --n_encoder_layers 2 --n_decoder_layers 2 --max_epochs 2 --save_dir checkpoints_quick

In [ ]:
# Full training example (adjust as needed)
# !python train_slt.py --device cuda --batch_size 16 --d_model 512 --n_encoder_layers 4 --n_decoder_layers 4 --max_epochs 30 --save_dir checkpoints_full

## 7) Save Results Back to Drive
Copy checkpoints/logs to Drive and create a zip archive for download/sharing.

In [ ]:
import shutil
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
run_name = f'slt_run_{timestamp}'

drive_out = f'/content/drive/MyDrive/slt_outputs/{run_name}'
os.makedirs(drive_out, exist_ok=True)

# Copy common checkpoint folders if they exist
for folder in ['checkpoints', 'checkpoints_quick', 'checkpoints_full']:
    if os.path.exists(folder):
        dst = os.path.join(drive_out, folder)
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(folder, dst)
        print('Copied:', folder, '->', dst)

# Zip outputs for convenience
zip_base = f'/content/{run_name}'
archive = shutil.make_archive(zip_base, 'zip', root_dir=drive_out)
print('Created zip:', archive)

# Also copy zip to Drive root output folder
final_zip = f'/content/drive/MyDrive/slt_outputs/{run_name}.zip'
shutil.copy2(archive, final_zip)
print('Saved zip to:', final_zip)